# Mini-Projeto ETL — IQVIA Data
## Fase 1: Ingestão — Camada Bronze

**Objetivo:** Carregar os arquivos brutos do dataset IQVIA e enviá-los para o bucket GCS na camada **Bronze**, sem transformação, registrando metadados de cada carga.

| Camada | Descrição |
|--------|-----------|
| **Bronze** | Dados brutos conforme recebidos, sem transformação |
| Silver | Dados limpos e padronizados (próxima fase) |
| Gold | Dados agregados prontos para consumo analítico |

**Bucket GCS:** `etl-iqvia-data-lake-augusto`  
**Prefix:** `bronze/iqvia/`

## 1. Instalação e Importação de Dependências

In [14]:
%pip install -r requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import os
import hashlib
import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from google.cloud import storage

## 2. Configurações do Pipeline

In [16]:
DATASET_PATH = Path("dataset")
GCS_PROJECT_ID = "proc-de-dados-iqvia-com-scd"
GCS_BUCKET_NAME = "etl-iqvia-data-lake-augusto"
GCS_BRONZE_PREFIX = "bronze/iqvia"
ACCEPTED_EXTENSIONS = [".xlsx", ".csv", ".parquet"]

LOAD_TIMESTAMP = datetime.now(timezone.utc)
LOAD_DATE_PARTITION = LOAD_TIMESTAMP.strftime("%Y/%m/%d")   # formato GCS (não alterar)
LOAD_DATE_DISPLAY = LOAD_TIMESTAMP.strftime("%d/%m/%Y")     # formato de exibição

print(f"Dataset local  : {DATASET_PATH.resolve()}")
print(f"Projeto GCP    : {GCS_PROJECT_ID}")
print(f"Bucket GCS     : {GCS_BUCKET_NAME}")
print(f"Prefixo Bronze : {GCS_BRONZE_PREFIX}")
print(f"Partição       : {LOAD_DATE_DISPLAY}")
print(f"Load timestamp : {LOAD_TIMESTAMP.strftime('%d/%m/%Y %H:%M:%S UTC')}")

Dataset local  : D:\analise-de-dados-devinhouse\modulo_02_arquitetura_e_visualizacao_de_dados\mini_projeto\dataset
Projeto GCP    : proc-de-dados-iqvia-com-scd
Bucket GCS     : etl-iqvia-data-lake-augusto
Prefixo Bronze : bronze/iqvia
Partição       : 18/03/2026
Load timestamp : 18/03/2026 21:36:55 UTC


## 3. Módulo de Utilitários

In [17]:
def compute_md5(file_path: Path) -> str:
    """Calcula o hash MD5 de um arquivo para verificação de integridade."""
    hash_md5 = hashlib.md5()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()


def get_file_size_mb(file_path: Path) -> float:
    """Retorna o tamanho do arquivo em MB."""
    return round(file_path.stat().st_size / (1024 * 1024), 4)


def list_source_files(dataset_path: Path, extensions: list) -> list[Path]:
    """Lista os arquivos de dados na pasta de origem filtrando por extensão."""
    files = [
        f for f in dataset_path.iterdir()
        if f.is_file() and f.suffix.lower() in extensions
    ]
    return sorted(files)


def build_gcs_destination_path(prefix: str, date_partition: str, file_name: str) -> str:
    """Monta o caminho de destino no GCS com particionamento por data.

    Estrutura: <prefix>/<YYYY>/<MM>/<DD>/<file_name>
    """
    return f"{prefix}/{date_partition}/{file_name}"

## 4. Módulo de Conexão com Google Cloud Storage

In [18]:
# Autenticação via Application Default Credentials (ADC).
# Execute previamente no terminal (requer Google Cloud SDK instalado):
#   gcloud auth application-default login
#   gcloud auth application-default set-quota-project proc-de-dados-iqvia-com-scd

def get_gcs_client() -> storage.Client:
    """Instancia e retorna o cliente do Google Cloud Storage."""
    client = storage.Client(project=GCS_PROJECT_ID)
    print(f"Conectado ao GCS | Projeto: {client.project}")
    return client


def get_bucket(client: storage.Client, bucket_name: str) -> storage.Bucket:
    """Recupera o bucket GCS e valida sua existência."""
    bucket = client.bucket(bucket_name)
    if not bucket.exists():
        raise ValueError(f"Bucket '{bucket_name}' nao encontrado ou sem permissao de acesso.")
    print(f"Bucket: gs://{bucket_name}")
    return bucket

## 5. Módulo de Upload (Extract → Load Bronze)

In [19]:
def upload_file_to_bronze(
    bucket: storage.Bucket,
    local_file: Path,
    gcs_destination: str,
    load_timestamp: datetime
) -> dict:
    """Faz upload de um arquivo local para a camada Bronze no GCS.

    Anexa metadados customizados ao blob (origem, hash MD5, timestamp de carga).

    Returns:
        dict: Registro de metadados do arquivo carregado.
    """
    file_md5 = compute_md5(local_file)
    file_size_mb = get_file_size_mb(local_file)

    blob = bucket.blob(gcs_destination)
    blob.metadata = {
        "source_file": local_file.name,
        "source_path": str(local_file.resolve()),
        "layer": "bronze",
        "pipeline": "iqvia-etl",
        "load_timestamp": load_timestamp.isoformat(),
        "md5_checksum": file_md5,
    }

    print(f"Enviando: {local_file.name} ({file_size_mb} MB)...")
    blob.upload_from_filename(str(local_file))
    print(f"  Salvo em: gs://{bucket.name}/{gcs_destination}")

    return {
        "file_name": local_file.name,
        "gcs_path": f"gs://{bucket.name}/{gcs_destination}",
        "size_mb": file_size_mb,
        "md5_checksum": file_md5,
        "load_timestamp": load_timestamp.isoformat(),
        "status": "success",
    }


def upload_ingestion_manifest(
    bucket: storage.Bucket,
    manifest: list[dict],
    prefix: str,
    date_partition: str,
    load_timestamp: datetime
) -> None:
    """Salva o manifesto de ingestão (JSON) no GCS para auditoria e rastreabilidade."""
    manifest_data = {
        "pipeline": "iqvia-etl",
        "layer": "bronze",
        "load_timestamp": load_timestamp.isoformat(),
        "total_files": len(manifest),
        "files": manifest,
    }
    manifest_json = json.dumps(manifest_data, indent=2, ensure_ascii=False)

    manifest_path = f"{prefix}/{date_partition}/_manifest.json"
    blob = bucket.blob(manifest_path)
    blob.upload_from_string(manifest_json, content_type="application/json")
    print(f"Manifesto salvo em: gs://{bucket.name}/{manifest_path}")

## 6. Execução do Pipeline de Ingestão

In [20]:
source_files = list_source_files(DATASET_PATH, ACCEPTED_EXTENSIONS)

if not source_files:
    raise FileNotFoundError(
        f"Nenhum arquivo encontrado em '{DATASET_PATH}' com extensoes: {ACCEPTED_EXTENSIONS}"
    )

print(f"{len(source_files)} arquivo(s) encontrado(s):")
for f in source_files:
    print(f"  {f.name} ({get_file_size_mb(f)} MB)")

2 arquivo(s) encontrado(s):
  filial-brick_sample.xlsx (0.0189 MB)
  MS_12_2022_sample.xlsx (0.255 MB)


In [21]:
# Leitura dos primeiros registros para validação — sem transformação (Bronze)
for file in source_files:
    print(f"\n{file.name}")
    print("-" * 60)

    if file.suffix.lower() in [".xlsx", ".xls"]:
        df_preview = pd.read_excel(file, nrows=5)
    elif file.suffix.lower() == ".csv":
        df_preview = pd.read_csv(file, nrows=5)
    elif file.suffix.lower() == ".parquet":
        df_preview = pd.read_parquet(file).head(5)

    print(f"Colunas ({len(df_preview.columns)}): {list(df_preview.columns)}")
    display(df_preview)


filial-brick_sample.xlsx
------------------------------------------------------------
Colunas (2): ['brick', 'Cód. Filial']


,brick,Cód. Filial
0,1 - SE,402
1,92 - CHAC.SANTO ANTONIO,454
2,181 - REGISTRO,423
3,257 - S.J.RIO PRETO-CENTRO,394
4,271 - S.J.RIO PRETO-V.S.MANOEL,417



MS_12_2022_sample.xlsx
------------------------------------------------------------
Colunas (6): ['BRICK', 'EAN', 'Cod Prod Catarinense', 'Tipo Informacao SI Bandeira CONCORRENTE Unidade', 'Tipo Informacao SO Bandeira CONCORRENTE Unidade', 'Tipo Informacao SO Bandeira PRECO POPULAR Unidade']


,BRICK,EAN,Cod Prod Catarinense,Tipo Informacao SI Bandeira CONCORRENTE Unidade,Tipo Informacao SO Bandeira CONCORRENTE Unidade,Tipo Informacao SO Bandeira PRECO POPULAR Unidade
0,1147 - CAMPO GRANDE - CENTRO,32689150,741013,NaN,0,5.0
1,1147 - CAMPO GRANDE - CENTRO,42110200,630693,NaN,7,5.0
2,1147 - CAMPO GRANDE - CENTRO,42176763,687607,NaN,4,NaN
3,1147 - CAMPO GRANDE - CENTRO,42277217,739189,7.0,85,23.0
4,1147 - CAMPO GRANDE - CENTRO,42355014,735367,NaN,2,1.0


In [22]:
gcs_client = get_gcs_client()
bucket = get_bucket(gcs_client, GCS_BUCKET_NAME)

Conectado ao GCS | Projeto: proc-de-dados-iqvia-com-scd
Bucket: gs://etl-iqvia-data-lake-augusto
Bucket: gs://etl-iqvia-data-lake-augusto


In [23]:
ingestion_manifest = []

for local_file in source_files:
    gcs_dest = build_gcs_destination_path(
        prefix=GCS_BRONZE_PREFIX,
        date_partition=LOAD_DATE_PARTITION,
        file_name=local_file.name
    )
    try:
        record = upload_file_to_bronze(
            bucket=bucket,
            local_file=local_file,
            gcs_destination=gcs_dest,
            load_timestamp=LOAD_TIMESTAMP
        )
    except Exception as exc:
        print(f"Erro ao enviar {local_file.name}: {exc}")
        record = {
            "file_name": local_file.name,
            "gcs_path": f"gs://{GCS_BUCKET_NAME}/{gcs_dest}",
            "size_mb": get_file_size_mb(local_file),
            "md5_checksum": compute_md5(local_file),
            "load_timestamp": LOAD_TIMESTAMP.isoformat(),
            "status": f"error: {exc}",
        }
    ingestion_manifest.append(record)

success = sum(1 for r in ingestion_manifest if r["status"] == "success")
print(f"\nUpload concluido: {success}/{len(ingestion_manifest)} arquivo(s) enviado(s) com sucesso.")

Enviando: filial-brick_sample.xlsx (0.0189 MB)...
  Salvo em: gs://etl-iqvia-data-lake-augusto/bronze/iqvia/2026/03/18/filial-brick_sample.xlsx
Enviando: MS_12_2022_sample.xlsx (0.255 MB)...
  Salvo em: gs://etl-iqvia-data-lake-augusto/bronze/iqvia/2026/03/18/MS_12_2022_sample.xlsx

Upload concluido: 2/2 arquivo(s) enviado(s) com sucesso.
  Salvo em: gs://etl-iqvia-data-lake-augusto/bronze/iqvia/2026/03/18/filial-brick_sample.xlsx
Enviando: MS_12_2022_sample.xlsx (0.255 MB)...
  Salvo em: gs://etl-iqvia-data-lake-augusto/bronze/iqvia/2026/03/18/MS_12_2022_sample.xlsx

Upload concluido: 2/2 arquivo(s) enviado(s) com sucesso.


In [24]:
# ─── STEP 5: Geração e upload do manifesto de ingestão ───────────────────────

upload_ingestion_manifest(
    bucket=bucket,
    manifest=ingestion_manifest,
    prefix=GCS_BRONZE_PREFIX,
    date_partition=LOAD_DATE_PARTITION,
    load_timestamp=LOAD_TIMESTAMP
)

Manifesto salvo em: gs://etl-iqvia-data-lake-augusto/bronze/iqvia/2026/03/18/_manifest.json


## 7. Relatório Final da Ingestão

In [25]:
df_report = pd.DataFrame(ingestion_manifest)

print("=" * 60)
print("RELATORIO DE INGESTAO — CAMADA BRONZE")
print("=" * 60)
print(f"  Pipeline  : IQVIA ETL")
print(f"  Bucket    : gs://{GCS_BUCKET_NAME}")
print(f"  Prefixo   : {GCS_BRONZE_PREFIX}/{LOAD_DATE_PARTITION}/")
print(f"  Timestamp : {LOAD_TIMESTAMP.strftime('%d/%m/%Y %H:%M:%S UTC')}")
print(f"  Total     : {len(ingestion_manifest)} arquivo(s)")
print(f"  Sucesso   : {df_report[df_report['status'] == 'success'].shape[0]}")
print(f"  Erros     : {df_report[df_report['status'] != 'success'].shape[0]}")
print("=" * 60)
display(df_report[["file_name", "size_mb", "md5_checksum", "gcs_path", "status"]])

RELATORIO DE INGESTAO — CAMADA BRONZE
  Pipeline  : IQVIA ETL
  Bucket    : gs://etl-iqvia-data-lake-augusto
  Prefixo   : bronze/iqvia/2026/03/18/
  Timestamp : 18/03/2026 21:36:55 UTC
  Total     : 2 arquivo(s)
  Sucesso   : 2
  Erros     : 0


,file_name,size_mb,md5_checksum,gcs_path,status
0,filial-brick_sample.xlsx,0.0189,9bf8e75fa138281c6c00d467f944b032,gs://etl-iqvia-data-lake-augusto/bronze/iqvia/...,success
1,MS_12_2022_sample.xlsx,0.2550,e50a113d8481b3e148da6067397f0fa0,gs://etl-iqvia-data-lake-augusto/bronze/iqvia/...,success


## 8. Verificação dos Arquivos no Bucket

Listagem dos blobs na partição de hoje para confirmar que os arquivos foram carregados.

In [26]:
prefix_filter = f"{GCS_BRONZE_PREFIX}/{LOAD_DATE_PARTITION}/"
blobs = list(gcs_client.list_blobs(GCS_BUCKET_NAME, prefix=prefix_filter))

print(f"Conteudo de gs://{GCS_BUCKET_NAME}/{prefix_filter}")
print("-" * 60)

if not blobs:
    print("  (nenhum arquivo encontrado)")
else:
    for blob in blobs:
        size_kb = round(blob.size / 1024, 2)
        updated = blob.updated.strftime("%d/%m/%Y %H:%M:%S UTC")
        print(f"  {blob.name.split('/')[-1]}  |  {size_kb} KB  |  {updated}")

print("\nIngestao Bronze finalizada. Pipeline pronto para a proxima fase (Silver).")

Conteudo de gs://etl-iqvia-data-lake-augusto/bronze/iqvia/2026/03/18/
------------------------------------------------------------
  MS_12_2022_sample.xlsx  |  261.14 KB  |  18/03/2026 21:36:59 UTC
  _manifest.json  |  0.78 KB  |  18/03/2026 21:36:59 UTC
  filial-brick_sample.xlsx  |  19.4 KB  |  18/03/2026 21:36:59 UTC

Ingestao Bronze finalizada. Pipeline pronto para a proxima fase (Silver).
